# Create Additional Features for Anomaly Detection

## About this Notebook

### Objective

Engineer and assemble derived features for anomaly detection in facility-level NO₂ predictions.

This notebook transforms model predictions, satellite-derived variables, and facility metadata into structured features that can be used to detect abnormal emissions behavior relative to expected atmospheric conditions.

---

### Inputs

- Facility-level predicted NO₂ values (from `PredictFacilityNO2`)
- Observed ground NO₂ (where available)
- Sentinel-2 and Sentinel-5 derived features
- Meteorological variables (e.g., wind speed, boundary layer height)
- Facility metadata and geographic identifiers

Primary data source:
`gs://msads-mba-capstone-team-1/`

---

### Outputs

- Feature-engineered dataset for anomaly modeling
- Residual-based features (observed vs predicted differences)
- Temporal trend and deviation metrics
- Spatially contextualized features
- Exported anomaly-ready dataset saved to GCS

Example output:
`gs://msads-mba-capstone-team-1/AnomalyData/facility_anomaly_features.csv`

---

### Feature Engineering Strategy

The notebook constructs features across several categories:

1. **Model Residual Features**
   - Predicted vs observed NO₂ differences
   - Absolute and squared residuals
   - Log-scale residuals

2. **Temporal Features**
   - Quarter-over-quarter change
   - Rolling means and deviations
   - Seasonal indicators

3. **Meteorological Controls**
   - Wind speed and direction components
   - Boundary layer height transformations
   - Log-transformed atmospheric variables

4. **Spatial Context**
   - Facility-level aggregation
   - Geographic identifiers (e.g., FIPS, tract-level metrics)
   - Neighborhood-level baselines

---

### Key Notes

- Residuals serve as the primary anomaly signal.
- Features are constructed to separate expected atmospheric variation from abnormal emissions patterns.
- Log transformations are used where distributions are skewed.
- All features must align temporally and spatially with model predictions.
- No test-set information should be used when computing baseline statistics.

---

### Pipeline Position

FusionNO2Model  
→ PredictFacilityNO2  
→ FeaturesAnomalyDetection  
→ Anomaly Modeling / Risk Scoring  

This notebook prepares the structured inputs required for downstream anomaly detection and emissions verification.

## Config

In [217]:
CONFIG = {
    
    "prediction_input_path":
        "gs://msads-mba-capstone-team-1/Data/Predictions/facility_predictions_2021_model_20260220_054518.csv",

    "facility_input_path":
        "gs://msads-mba-capstone-team-1/Data/TrainingData/nei_2021_IL_nox_for_model.csv",

    "s5_statewide_base":
        "msads-mba-capstone-team-1/Data/TrainingData/S5_Statewide",

    "ee_export_base":
        "gs://msads-mba-capstone-team-1/Data/TrainingData/",

    "background_features_output_path":
        "gs://msads-mba-capstone-team-1/Data/Predictions/facility_background_no2.csv",

    "s2s5_features_output_path":
        "gs://msads-mba-capstone-team-1/Data/TrainingData/facility_s2s5_values.csv",

    "anomaly_model_features_output_path":
        "gs://msads-mba-capstone-team-1/Data/Predictions/anomaly_model_features.csv",
    
    "ee_project_name": 
        "msads-mba-autumn-2025-team-1",

    "year": 2021,
    "quarters": ["Q1", "Q2", "Q3", "Q4"],
}

ee_project_name = "msads-mba-autumn-2025-team-1"
bucket_name = "msads-mba-capstone-team-1"
gcs_folder_s2 = "Data/TrainingData/S2_Statewide"
gcs_folder_s5 = "Data/TrainingData/S5_Statewide"

## Setup

In [164]:
import numpy as np
import pandas as pd
import gcsfs
import rasterio
from rasterio.windows import from_bounds
from pyproj import Transformer
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.neighbors import BallTree

In [40]:
fs = gcsfs.GCSFileSystem()

with fs.open(CONFIG["prediction_input_path"]) as f:
    predictions_df = pd.read_csv(f)

with fs.open(CONFIG["facility_input_path"]) as f:
    facilities = pd.read_csv(f)

print(predictions_df.shape)
print(facilities.shape)

(9276, 4)
(2319, 11)


In [41]:
facilities = facilities.rename(columns={
    "eis facility id": "facility_id",
    "total emissions": "reported_emissions",
    "primary naics code": "naics_code",
    "site latitude": "lat",
    "site longitude": "lon"
})

facility_df = predictions_df.merge(
    facilities,
    on="facility_id",
    how="left"
)

facility_df = facility_df.dropna(subset=["predicted_no2"])

print("Facilities:", facility_df["facility_id"].nunique())

Facilities: 2287


## New Feature Creation

### Computing background NO2 (5 km radius)

For each facility, get the NO2 concentration for a 5km radius - this is the surrounding No2 that the facility did not contribute to.

In [6]:
def compute_background_s5(tile_paths, lon, lat,
                          radius_m=5000):

    chips = []

    for path in tile_paths:

        with rasterio.open(f"/vsigs/{path}") as src:

            transformer = Transformer.from_crs(
                "EPSG:4326",
                src.crs,
                always_xy=True
            )

            x, y = transformer.transform(lon, lat)

            left   = x - radius_m
            right  = x + radius_m
            bottom = y - radius_m
            top    = y + radius_m

            bounds = src.bounds

            intersects = not (
                right < bounds.left or
                left > bounds.right or
                top < bounds.bottom or
                bottom > bounds.top
            )

            if not intersects:
                continue

            window = from_bounds(
                left, bottom, right, top,
                src.transform
            )

            chip = src.read(
                window=window,
                boundless=True,
                fill_value=np.nan
            )

            chips.append(chip)

    if len(chips) == 0:
        return np.nan

    chip = np.maximum.reduce(chips)

    band = chip[0]  # single S5 band

    # Compute circular mask
    height, width = band.shape
    center_x = width // 2
    center_y = height // 2

    pixel_size = 10  # S5 statewide resolution
    radius_pixels = radius_m / pixel_size

    yy, xx = np.ogrid[:height, :width]
    distance = np.sqrt((xx - center_x)**2 + (yy - center_y)**2)

    mask = distance <= radius_pixels

    values = band[mask]

    return np.nanmean(values)

In [7]:
fs = gcsfs.GCSFileSystem()

facility_df["background_s5"] = np.nan

for quarter in CONFIG["quarters"]:

    print(f"\nComputing background for {quarter}")

    s5_tiles = fs.glob(
        f"{CONFIG['s5_statewide_base']}/S5_Statewide_{CONFIG['year']}{quarter}*.tif"
    )

    subset = facility_df[facility_df["quarter"] == quarter]

    for idx, row in tqdm(subset.iterrows(),
                         total=len(subset)):

        bg = compute_background_s5(
            s5_tiles,
            row["lon"],
            row["lat"],
            radius_m=5000
            #outer_radius=10000,
            #inner_radius=1000
        )

        facility_df.loc[idx, "background_s5"] = bg


Computing background for Q1


100%|██████████| 2287/2287 [15:50<00:00,  2.41it/s]



Computing background for Q2


100%|██████████| 2287/2287 [16:22<00:00,  2.33it/s]



Computing background for Q3


100%|██████████| 2287/2287 [17:25<00:00,  2.19it/s]



Computing background for Q4


100%|██████████| 2287/2287 [16:29<00:00,  2.31it/s]


In [10]:
# Save Results
facility_df.to_csv(
    "gs://msads-mba-capstone-team-1/Data/Predictions/facility_background_no2.csv",
    index=False
)

print("Facility DF index saved successfully.")

Facility DF index saved successfully.


In [165]:
fs = gcsfs.GCSFileSystem()

with fs.open(CONFIG["background_features_output_path"]) as f:
    facility_df = pd.read_csv(f)

### Facilities within 5 km radius

Calculate the number of facilities within a 5 km radius + sum their emissions

In [166]:
coords = np.radians(
    facility_df[["lat", "lon"]].values
)

tree = BallTree(coords, metric="haversine")

In [167]:
earth_radius_km = 6371
radius = 5 / earth_radius_km  # radians

In [168]:
facility_counts = []

for i in range(len(facility_df)):

    ind = tree.query_radius(
        coords[i:i+1],
        r=radius
    )[0]

    # Remove itself
    ind = ind[ind != i]

    facility_counts.append(len(ind))

facility_df["facility_density_5km"] = facility_counts

### Nearby Emissions Sum

In [169]:
nearby_emissions = []

for i in range(len(facility_df)):

    ind = tree.query_radius(
        coords[i:i+1],
        r=radius
    )[0]

    # Remove itself
    ind = ind[ind != i]

    total = facility_df.iloc[ind]["reported_emissions"].sum()

    nearby_emissions.append(total)

facility_df["nearby_emissions_5km"] = nearby_emissions

In [170]:
facility_df["log_nearby_emissions_5km"] = np.log1p(
    facility_df["nearby_emissions_5km"]
)

### Wind Speed and Direction + Boundary Layer Height

All of these factors contribute to the dispersion of No2 in the atmosphere

In [171]:
import cdsapi
import xarray as xr
import numpy as np
import pandas as pd
import ee

In [180]:
ee.Initialize(project=CONFIG['ee_project_name'])

# Get Illinois boundary
states = ee.FeatureCollection("TIGER/2018/States")
illinois = states.filter(ee.Filter.eq("NAME", "Illinois")).geometry()

In [173]:
features = []

for _, row in facility_df.iterrows():

    feature = ee.Feature(
        ee.Geometry.Point(row["lon"], row["lat"]),
        {
            "facility_id": row["facility_id"],
            "quarter": row["quarter"]
        }
    )

    features.append(feature)

fc = ee.FeatureCollection(features)

In [174]:
#Wind Speed and Direction + Boundary Layer Height Extraction
all_results = []

quarter_groups = facility_df.groupby("quarter")

for quarter, df_q in quarter_groups:

    print(f"\nProcessing {quarter}...")

    # -----------------------------
    # Define quarter dates
    # -----------------------------
    if quarter == "Q1":
        start = f"{CONFIG['year']}-01-01"
        end   = f"{CONFIG['year']}-03-31"
    elif quarter == "Q2":
        start = f"{CONFIG['year']}-04-01"
        end   = f"{CONFIG['year']}-06-30"
    elif quarter == "Q3":
        start = f"{CONFIG['year']}-07-01"
        end   = f"{CONFIG['year']}-09-30"
    elif quarter == "Q4":
        start = f"{CONFIG['year']}-10-01"
        end   = f"{CONFIG['year']}-12-31"

    # -----------------------------
    # Build ERA5 quarterly mean image
    # Compute wind speed BEFORE averaging
    # -----------------------------
    era5 = (
        ee.ImageCollection("ECMWF/ERA5/HOURLY")
        .filterDate(start, end)
        .select([
            "u_component_of_wind_10m",
            "v_component_of_wind_10m",
            "boundary_layer_height"
        ])
        .map(lambda img: img.addBands(
            img.expression(
                "sqrt(u*u + v*v)",
                {
                    "u": img.select("u_component_of_wind_10m"),
                    "v": img.select("v_component_of_wind_10m")
                }
            ).rename("wind_speed")
        ))
        .mean()
        .select([
            "u_component_of_wind_10m",
            "v_component_of_wind_10m",
            "boundary_layer_height",
            "wind_speed"
        ])
    )

    # -----------------------------
    # Build FeatureCollection for this quarter
    # -----------------------------
    features = []

    for _, row in df_q.iterrows():

        features.append(
            ee.Feature(
                ee.Geometry.Point(row["lon"], row["lat"]),
                {
                    "facility_id": row["facility_id"],
                    "quarter": quarter,
                    "year": CONFIG["year"]
                }
            )
        )

    fc = ee.FeatureCollection(features)

    # -----------------------------
    # Sample all facilities at once
    # -----------------------------
    sampled = era5.sampleRegions(
        collection=fc,
        scale=30000
    )

    results = sampled.getInfo()

    for f in results["features"]:
        all_results.append(f["properties"])

    print(f"{quarter} complete.")


Processing Q1...
Q1 complete.

Processing Q2...
Q2 complete.

Processing Q3...
Q3 complete.

Processing Q4...
Q4 complete.


In [175]:
wind_df = pd.DataFrame(all_results)

In [176]:
wind_df.head()

,boundary_layer_height,facility_id,quarter,u_component_of_wind_10m,v_component_of_wind_10m,wind_speed,year
0,494.547333,558811,Q1,0.452169,-0.171152,4.366572,2021
1,485.923004,558911,Q1,0.800267,-0.111828,4.753916,2021
2,485.923004,559311,Q1,0.800267,-0.111828,4.753916,2021
3,485.538635,559511,Q1,0.803628,0.035672,4.550055,2021
4,447.287079,559711,Q1,0.554807,-0.162754,3.971008,2021


In [177]:
facility_df = facility_df.merge(
    wind_df,
    on=["facility_id","quarter"],
    how="left"
)

In [178]:
facility_df["wind_speed"] = np.sqrt(
    facility_df["u_component_of_wind_10m"]**2 +
    facility_df["v_component_of_wind_10m"]**2
)

facility_df["wind_dir"] = (
    270 - np.degrees(
        np.arctan2(
            facility_df["v_component_of_wind_10m"],
            facility_df["u_component_of_wind_10m"]
        )
    )
) % 360

facility_df["wind_dir_rad"] = np.deg2rad(
    facility_df["wind_dir"]
)

facility_df["wind_u_dir"] = np.cos(
    facility_df["wind_dir_rad"]
)

facility_df["wind_v_dir"] = np.sin(
    facility_df["wind_dir_rad"]
)

facility_df["log_blh"] = np.log1p(
    facility_df["boundary_layer_height"]
)

facility_df["log_wind_speed"] = np.log1p(
    facility_df["wind_speed"]
)

In [179]:
facility_df[[
    "wind_speed",
    "wind_dir",
    "log_blh"
]].describe()

,wind_speed,wind_dir,log_blh
count,9148.000000,9148.000000,9148.000000
mean,0.860917,235.599739,6.316600
std,0.344438,32.818857,0.159580
min,0.108040,165.129238,5.505220
25%,0.637130,212.564715,6.221583
50%,0.811162,226.207632,6.287208
75%,1.069273,253.930905,6.410233
max,2.011941,332.759035,6.693742


### Get S2 and S5 Value Features

In [207]:
ee.Initialize(project=CONFIG['ee_project_name'])

In [204]:
# Controls
YEAR = 2021
QUARTERS = ["Q1","Q2","Q3","Q4"]

START_INDEX = 0
END_INDEX = None
BATCH_SIZE = 200

In [203]:
# Subset Facilities
facility_subset = facility_df.iloc[START_INDEX:END_INDEX].reset_index(drop=True)
print("Facilities:", len(facility_subset))

Facilities: 9148


In [205]:
# Convert to EE Features
def df_to_ee(df):
    features = []
    for _, row in df.iterrows():
        geom = ee.Geometry.Point([row["lon"], row["lat"]])
        feat = ee.Feature(geom, {
            "facility_id": str(row["facility_id"])
        })
        features.append(feat)
    return ee.FeatureCollection(features)

In [214]:
from tqdm.auto import tqdm
import pandas as pd
import ee

# ---------------------------------
# SETTINGS
# ---------------------------------
YEAR = 2021
BATCH_SIZE = 200
BUFFER_METERS = 1000
SCALE = 1000

# ---------------------------------
# Helper: Date Mapping
# ---------------------------------
def get_dates(year, quarter):
    dates = {
        "Q1": (f"{year}-01-01", f"{year}-03-31"),
        "Q2": (f"{year}-04-01", f"{year}-06-30"),
        "Q3": (f"{year}-07-01", f"{year}-09-30"),
        "Q4": (f"{year}-10-01", f"{year}-12-31"),
    }
    return dates[quarter]

# ---------------------------------
# Updated S5 Builder
# ---------------------------------
def get_s5_quarter(year, quarter):
    start, end = get_dates(year, quarter)
    return (
        ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")
        .filterDate(start, end)
        .select("tropospheric_NO2_column_number_density")
        .mean()
    )

# ---------------------------------
# Updated S2_HARMONIZED Builder
# ---------------------------------
def get_s2_quarter(year, quarter):
    start, end = get_dates(year, quarter)

    return (
        ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(lambda img: img.select([
            "B1","B2","B3","B4","B5","B6",
            "B7","B8","B8A","B9","B11","B12"
        ]))
        .median()
    )

# ---------------------------------
# Main Extraction Loop
# ---------------------------------
all_rows = []

quarters = QUARTERS  # assumes facility_df has quarter column

for quarter in quarters:

    print(f"\nProcessing {quarter}")

    df_q = facility_df[facility_df["quarter"] == quarter].reset_index(drop=True)

    if len(df_q) == 0:
        continue

    s5_img = get_s5_quarter(YEAR, quarter)
    s2_img = get_s2_quarter(YEAR, quarter)
    combined = s5_img.addBands(s2_img)

    for batch_start in tqdm(range(0, len(df_q), BATCH_SIZE)):

        batch_end = min(batch_start + BATCH_SIZE, len(df_q))
        batch_df = df_q.iloc[batch_start:batch_end]

        features = []
        for _, row in batch_df.iterrows():
            geom = ee.Geometry.Point([
                row["lon"],
                row["lat"]
            ])
            feat = ee.Feature(geom, {
                "facility_id": str(row["facility_id"]),
                "quarter": quarter
            })
            features.append(feat)

        facilities_ee = ee.FeatureCollection(features)

        # Buffer to reduce noise
        buffered = facilities_ee.map(lambda f: f.buffer(BUFFER_METERS))

        sampled = combined.reduceRegions(
            collection=buffered,
            reducer=ee.Reducer.mean(),
            scale=SCALE
        )

        results = sampled.getInfo()

        for feature in results["features"]:

            props = feature["properties"]

            row_out = {
                "facility_id": props.get("facility_id"),
                "quarter": props.get("quarter")
            }

            for key, value in props.items():
                if key not in ["facility_id", "quarter"]:
                    row_out[key] = value

            all_rows.append(row_out)

sat_quarter_df = pd.DataFrame(all_rows)

print("\nExtraction complete.")
print("Rows extracted:", len(sat_quarter_df))
print(sat_quarter_df.head())


Processing Q1


  0%|          | 0/12 [00:00<?, ?it/s]


Processing Q2


  0%|          | 0/12 [00:00<?, ?it/s]

httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.



Processing Q3


  0%|          | 0/12 [00:00<?, ?it/s]


Processing Q4


  0%|          | 0/12 [00:00<?, ?it/s]


Extraction complete.
Rows extracted: 9148
  facility_id quarter           B1          B11          B12           B2  \
0      558811      Q1  1489.995037  1910.330025  1337.736973  1235.810174   
1      558911      Q1  1937.835775  1831.567155  1453.986569  1742.241148   
2      559311      Q1  1776.347305  1562.165868  1277.819760  1612.447305   
3      559511      Q1  1936.400836  1551.806452  1222.603345  1647.662485   
4      559711      Q1  2643.635135  2265.539312  1676.034398  2372.106265   

            B3           B4           B5           B6           B7  \
0  1089.764268  1187.736973  1282.166253  1371.166253  1489.197270   
1  1598.328449  1702.018926  1808.888889  1960.065324  2051.194750   
2  1424.494611  1481.209581  1573.222754  1698.909581  1769.886228   
3  1466.494624  1531.821386  1625.905018  1732.658901  1805.468937   
4  2201.542998  2407.328624  2593.892506  2875.085995  2980.243857   

            B8          B8A           B9  \
0  1535.481390  1667.178660  

In [218]:
# Save Results
sat_quarter_df.to_csv(
    CONFIG["s2s5_features_output_path"],
    index=False
)

print("Intermediate satellite features saved successfully.")

Intermediate satellite features saved successfully.


In [219]:
sat_quarter_df.columns

Index(['facility_id', 'quarter', 'B1', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5',
       'B6', 'B7', 'B8', 'B8A', 'B9',
       'tropospheric_NO2_column_number_density'],
      dtype='object')

In [221]:
facility_df["facility_id"] = facility_df["facility_id"].astype(str)
sat_quarter_df["facility_id"] = sat_quarter_df["facility_id"].astype(str)

In [222]:
facility_df = facility_df.merge(
    sat_quarter_df,
    on=["facility_id", "quarter"],
    how="left"
)

In [212]:
# --------------------------
# SETTINGS
# --------------------------
YEAR = 2021
TEST_QUARTER = "Q1"
TEST_SIZE = 10
BUFFER_METERS = 1000
SCALE = 1000

# --------------------------
# Helper: Quarter Date Mapping
# --------------------------
def get_dates(year, quarter):
    dates = {
        "Q1": (f"{year}-01-01", f"{year}-03-31"),
        "Q2": (f"{year}-04-01", f"{year}-06-30"),
        "Q3": (f"{year}-07-01", f"{year}-09-30"),
        "Q4": (f"{year}-10-01", f"{year}-12-31"),
    }
    return dates[quarter]

# --------------------------
# Build S5 Image
# --------------------------
start, end = get_dates(YEAR, TEST_QUARTER)

s5_img = (
    ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")
    .filterDate(start, end)
    .select("tropospheric_NO2_column_number_density")
    .mean()
)

# --------------------------
# Build S2 Image
# --------------------------
s2_img = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterDate(start, end)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
    .map(lambda img: img.select([
        "B1","B2","B3","B4","B5","B6",
        "B7","B8","B8A","B9","B11","B12"
    ]))
    .median()
)

# Combine
combined = s5_img.addBands(s2_img)

# --------------------------
# Subset Facilities
# --------------------------
test_df = facility_df.head(TEST_SIZE).copy()

print(f"Testing {len(test_df)} facilities for {TEST_QUARTER}")

# --------------------------
# Convert to EE FeatureCollection
# --------------------------
features = []

for _, row in test_df.iterrows():
    geom = ee.Geometry.Point([row["lon"], row["lat"]])
    feat = ee.Feature(geom, {
        "facility_id": str(row["facility_id"])
    })
    features.append(feat)

facilities_ee = ee.FeatureCollection(features)

# Buffer (recommended)
buffered = facilities_ee.map(lambda f: f.buffer(BUFFER_METERS))

# --------------------------
# Sample
# --------------------------
sampled = combined.reduceRegions(
    collection=buffered,
    reducer=ee.Reducer.mean(),
    scale=SCALE
)

# Pull to Python
results = sampled.getInfo()

rows = []
for feature in results["features"]:
    rows.append(feature["properties"])

test_sat_df = pd.DataFrame(rows)

# --------------------------
# Diagnostics
# --------------------------
print("\nReturned columns:")
print(test_sat_df.columns.tolist())

print("\nPreview:")
print(test_sat_df.head())

print("\nNull counts:")
print(test_sat_df.isna().sum())

Testing 10 facilities for Q1

Returned columns:
['B1', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'facility_id', 'tropospheric_NO2_column_number_density']

Preview:
            B1          B11          B12           B2           B3  \
0  1489.995037  1910.330025  1337.736973  1235.810174  1089.764268   
1  1937.665037  1831.339242  1453.843521  1742.128973  1598.195599   
2  1776.532413  1562.336735  1277.959184  1612.680072  1424.750900   
3  1936.813995  1552.482656  1223.144737  1648.028708  1466.928828   
4  2644.953144  2265.959309  1676.680641  2373.016030  2202.608508   

            B4           B5           B6           B7           B8  \
0  1187.736973  1282.166253  1371.166253  1489.197270  1535.481390   
1  1701.842298  1808.711491  1959.930318  2051.061736  2020.846577   
2  1481.484394  1573.531813  1699.337335  1770.338535  1741.441777   
3  1532.404306  1626.565789  1733.436603  1806.294258  1783.395933   
4  2408.697904  2595.241060  2876.1855

### Feature Engineering

In [235]:
annual_facility_df = (
    facility_df
    .groupby("facility_id")
    .mean(numeric_only=True)
    .reset_index()
)

In [236]:
#We will be predicting the contribution of the facility to the No2 so we subtract the background no2
annual_facility_df["facility_contribution"] = (
    annual_facility_df["predicted_no2"] -
    annual_facility_df["background_s5"]
)

annual_facility_df = annual_facility_df[
    annual_facility_df["facility_contribution"].notna()
]

In [237]:
epsilon = 1  # small constant

annual_facility_df["log_contribution"] = np.log1p(
    annual_facility_df["facility_contribution"].clip(lower=0)
)

In [238]:
annual_facility_df["log_predicted_no2"] = np.log1p(
    annual_facility_df["predicted_no2"]
)

In [239]:
annual_facility_df["log_emissions"] = np.log1p(
    annual_facility_df["reported_emissions"]
)

In [241]:
# Industry Context
# Extract 2-digit NAICS sector
annual_facility_df["naics_2digit"] = (
    annual_facility_df["naics_code"]
    .astype(str)
    .str[:2]
)

# Sector-level emission statistics
sector_stats = (
    annual_facility_df
    .groupby("naics_2digit")["reported_emissions"]
    .agg(["mean", "std"])
    .reset_index()
)

sector_stats.columns = [
    "naics_2digit",
    "sector_emission_mean",
    "sector_emission_std"
]

annual_facility_df = annual_facility_df.merge(
    sector_stats,
    on="naics_2digit",
    how="left"
)

# Sector-relative emission score
annual_facility_df["sector_emission_z"] = (
    annual_facility_df["reported_emissions"] - annual_facility_df["sector_emission_mean"]
) / annual_facility_df["sector_emission_std"]

In [242]:
# Percentile Rankings
annual_facility_df["emission_percentile"] = annual_facility_df["reported_emissions"].rank(pct=True)
annual_facility_df["no2_percentile"] = annual_facility_df["predicted_no2"].rank(pct=True)

In [159]:
len(annual_facility_df)

2287

## Save Data

In [243]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
annual_facility_df.head()

,facility_id,predicted_no2_log,predicted_no2,naics_code,lat,lon,GEOID10,reported_emissions,log_emissions,background_s5,facility_density_5km,nearby_emissions_5km,log_nearby_emissions_5km,boundary_layer_height,u_component_of_wind_10m,v_component_of_wind_10m,wind_speed,year,wind_dir,wind_dir_rad,wind_u_dir,wind_v_dir,log_blh,log_wind_speed,B1,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,B9,tropospheric_NO2_column_number_density,facility_contribution,log_contribution,log_predicted_no2,naics_2digit,sector_emission_mean,sector_emission_std,sector_emission_z,emission_percentile,no2_percentile
0,10691811,2.693790,14.795835,325180.0,41.512055,-87.617364,1.703183e+11,28.678680,3.390429,0.000080,79.0,973.928724,6.882364,535.416290,0.673510,0.530915,0.926306,2021.0,230.607684,4.024863,-0.583318,-0.702578,6.282809,0.638136,1772.995664,2022.568780,1433.859898,1569.942285,1450.444378,1454.033941,1626.181519,2133.683164,2358.417315,2281.009420,2516.132626,877.382177,0.000082,14.795755,2.759741,2.759746,32,24.283674,163.466845,0.026886,0.917796,0.996502
1,10692111,2.684778,14.669226,331110.0,41.656474,-87.625847,1.703182e+11,83.787327,4.440146,0.000076,99.0,2457.479937,7.807299,535.416290,0.673510,0.530915,0.926306,2021.0,230.607684,4.024863,-0.583318,-0.702578,6.282809,0.638136,1569.684444,1473.620416,1039.985330,1263.815556,1100.030715,1054.038967,1203.490984,1673.444835,1901.188417,1803.973258,2030.524297,686.915495,0.000105,14.669150,2.751694,2.751699,33,6.169623,31.847240,2.437188,0.960210,0.963271
2,10692311,2.605705,13.574336,332811.0,42.008329,-87.957168,1.703177e+11,5.168654,1.819481,0.000055,271.0,715.466854,6.574332,575.375710,0.687042,0.388107,0.849376,2021.0,240.118891,4.190865,-0.465104,-0.800729,6.347190,0.604490,2260.650652,2501.958778,1799.944395,2090.338523,1991.734282,2118.651246,2212.331554,2480.359282,2654.527580,2540.815391,2772.882562,1046.813167,0.000108,13.574281,2.679258,2.679262,33,6.169623,31.847240,-0.031430,0.738959,0.676432
3,10692411,2.680742,14.619081,517122.0,41.814630,-87.605669,1.703139e+11,0.086040,0.082538,0.000072,131.0,1053.513524,6.960835,520.401962,0.633031,0.438434,0.881732,2021.0,232.660595,4.060693,-0.524914,-0.679927,6.254914,0.611176,1921.280257,2056.036522,1562.329921,1673.908313,1539.131418,1537.451100,1678.256877,2068.093368,2281.314639,2147.246180,2412.638753,875.118582,0.000127,14.619009,2.748489,2.748493,51,1.633558,2.864471,-0.540246,0.102755,0.956712
4,10715911,2.520177,12.430801,324121.0,42.275124,-89.573565,1.717700e+11,4.709520,1.742135,0.000030,31.0,167.440178,5.126581,591.480972,0.591742,0.385130,0.746297,2021.0,242.478799,4.232053,-0.436997,-0.822424,6.360142,0.549755,2175.387910,2327.748937,1624.162515,1933.416464,1865.039642,1918.461422,2113.067284,2740.874089,3061.104648,2929.777187,3227.407807,1081.274301,0.000030,12.430771,2.597548,2.597551,32,24.283674,163.466845,-0.119744,0.722344,0.235899


In [244]:
annual_facility_df.columns

Index(['facility_id', 'predicted_no2_log', 'predicted_no2', 'naics_code',
       'lat', 'lon', 'GEOID10', 'reported_emissions', 'log_emissions',
       'background_s5', 'facility_density_5km', 'nearby_emissions_5km',
       'log_nearby_emissions_5km', 'boundary_layer_height',
       'u_component_of_wind_10m', 'v_component_of_wind_10m', 'wind_speed',
       'year', 'wind_dir', 'wind_dir_rad', 'wind_u_dir', 'wind_v_dir',
       'log_blh', 'log_wind_speed', 'B1', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5',
       'B6', 'B7', 'B8', 'B8A', 'B9', 'tropospheric_NO2_column_number_density',
       'facility_contribution', 'log_contribution', 'log_predicted_no2',
       'naics_2digit', 'sector_emission_mean', 'sector_emission_std',
       'sector_emission_z', 'emission_percentile', 'no2_percentile'],
      dtype='object')

In [245]:
# Save Results
annual_facility_df.to_csv(
    CONFIG["anomaly_model_features_output_path"],
    index=False
)

print("Anomaly Features DF index saved successfully.")

Anomaly Features DF index saved successfully.
